# 00 · Levantar el entorno (Docker + Spark) desde el notebook

Este notebook **corre en tu computadora**, no dentro del contenedor: es el que ejecuta `docker compose`.
Desde aquí se levanta el cluster de Spark, se verifica que esté sano y —si quieres— se ejecuta el análisis
completo sin abrir nada más.

> **Por qué son dos notebooks.** `Parte1_EDA_KDD.ipynb` corre *dentro* del contenedor, así que no puede
> levantar el contenedor que lo contiene (el huevo y la gallina). Este notebook vive afuera y se encarga de eso.

**Requisitos en tu máquina:**

- Docker Desktop (o Docker Engine + plugin compose) **encendido**.
- La carpeta del repo debe tener `docker-compose.yml` y la data en `data/<año>/`. No hace falta ningún Dockerfile.
- Para abrir este notebook: Jupyter (`pip install notebook` y luego `jupyter notebook`) o VS Code con la
  extensión de Jupyter. No necesitas Java, Spark ni PySpark instalados: eso vive en la imagen.

**Orden:** ejecutar las celdas de arriba hacia abajo (`Run All` funciona).

## 1. Utilidad para correr comandos

In [ ]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

PROY = Path.cwd()          # debe ser la carpeta del repo (donde está docker-compose.yml)
print("Carpeta del proyecto:", PROY)

def sh(cmd, check=True, mostrar=True):
    # Ejecuta un comando y muestra su salida en vivo, igual que en la terminal.
    if mostrar:
        print(f"$ {cmd}\n")
    p = subprocess.Popen(cmd, shell=True, cwd=PROY, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    salida = []
    for linea in p.stdout:
        salida.append(linea)
        if mostrar:
            print(linea, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"El comando falló (código {p.returncode}): {cmd}")
    return "".join(salida), p.returncode

## 2. Comprobar que Docker está instalado y corriendo

In [ ]:
assert (PROY / "docker-compose.yml").exists(), \
    f"No veo docker-compose.yml en {PROY}. Abre este notebook desde la carpeta del repo."

# 1) ¿Está el CLI de Docker?
_, rc = sh("docker --version", check=False)
if rc != 0:
    raise SystemExit("No encuentro el comando 'docker'. Instala Docker Desktop (Windows/macOS) "
                     "o docker.io + docker-compose-plugin (Linux), y reinicia el kernel.")

# 2) ¿Está el plugin compose?
_, rc = sh("docker compose version", check=False)
if rc != 0:
    raise SystemExit("Tienes Docker pero no el plugin 'compose'. En Linux: sudo apt install docker-compose-plugin")

# 3) ¿El daemon responde? (sin --format: las comillas simples no funcionan en cmd de Windows)
info, rc = sh("docker info", check=False, mostrar=False)
if rc != 0:
    bajo = info.lower()
    if "permission denied" in bajo:
        ayuda = "Linux: sudo usermod -aG docker $USER  (luego cierra sesión y vuelve a entrar)"
    elif "wsl" in bajo or "pipe" in bajo:
        ayuda = "Windows: abre Docker Desktop y activa Settings > Resources > WSL integration"
    else:
        ayuda = ("Windows/macOS: abre Docker Desktop y espera a que diga 'Engine running'.\n"
                 "Linux: sudo systemctl start docker")
    print(info[-1500:])
    raise SystemExit(f"Docker está instalado pero el daemon no responde.\n{ayuda}")

# 4) Resumen del entorno
for clave in ("Server Version:", "Operating System:", "OSType:", "Architecture:", "CPUs:", "Total Memory:"):
    for linea in info.splitlines():
        if linea.strip().startswith(clave):
            print(linea.strip())
            break
print("\nDocker OK.")

## 3. Revisar que la data esté en su sitio

In [ ]:
DATA = PROY / "data"
anios = sorted(d.name for d in DATA.glob("*") if d.is_dir() and d.name.isdigit()) if DATA.exists() else []
if anios:
    for a in anios:
        archivos = list((DATA / a).glob("*.json"))
        mb = sum(f.stat().st_size for f in archivos) / 1024**2
        print(f"  data/{a}: {len(archivos)} archivos, {mb:,.0f} MB")
elif DATA.exists() and list(DATA.glob("*.json")):
    print(f"  data/: {len(list(DATA.glob('*.json')))} archivos JSON (formato plano, también sirve)")
else:
    print("data/ está vacía. Descarga los JSON antes de seguir:\n"
          "    pip install -r requirements.txt\n"
          "    python3 download/download.py")

## 4. Levantar el cluster

No hay build: se usa la imagen oficial `quay.io/jupyter/pyspark-notebook:spark-3.5.0` tal cual. La primera vez
Docker la descarga (~2 GB, unos minutos); después arranca en segundos. `-d` la deja corriendo en segundo plano.

In [ ]:
sh("docker compose up -d")
_ = sh("docker compose ps")

## 5. Verificar que el cluster está sano

El master expone su estado en `http://localhost:8080/json/`. Se consulta hasta que aparezca al menos un
worker **ALIVE**: si el worker no se registra, los jobs se quedarían esperando para siempre.

In [ ]:
def estado_cluster(intentos=30, espera=2):
    for i in range(intentos):
        try:
            with urllib.request.urlopen("http://localhost:8080/json/", timeout=3) as r:
                est = json.load(r)
            vivos = [w for w in est.get("workers", []) if w.get("state") == "ALIVE"]
            if vivos:
                return est, vivos
        except Exception:
            pass
        time.sleep(espera)
    return None, []

est, vivos = estado_cluster()
if not vivos:
    print("El master no reporta workers. Revisa los logs:")
    sh("docker compose logs --tail=40 spark-worker", check=False)
else:
    cores = sum(w["cores"] for w in vivos)
    mem = sum(w["memory"] for w in vivos) / 1024
    print(f"Master : {est['url']}  ({est['status']})")
    print(f"Workers: {len(vivos)} ALIVE | {cores} cores | {mem:.1f} GB para executors")
    print("\nTodo listo. Abre:  http://localhost:8888  ->  Parte1_EDA_KDD.ipynb  ->  Run All")

## 6. (Opcional) Ejecutar el análisis completo sin abrir Jupyter

Hay dos formas de correr el análisis; **elige una**:

- **Interactiva (lo normal):** no toques esta celda. Abre <http://localhost:8888>, entra a
  `Parte1_EDA_KDD.ipynb` y dale `Run All`. Ves cada gráfico y cada tabla mientras se calculan.
- **De corrido (para entregar):** pon `EJECUTAR_TODO = True`. Esta celda corre el notebook **dentro del
  contenedor** y guarda una copia con todas las salidas en `Parte1_EDA_KDD_ejecutado.ipynb`, sin que tengas
  que abrir Jupyter. Con los tres años puede tardar; el timeout está en 2 horas.

Está en `False` por defecto para que un `Run All` de este notebook solo deje el cluster listo.

In [ ]:
EJECUTAR_TODO = False

if EJECUTAR_TODO:
    sh("docker compose exec -T jupyter jupyter nbconvert "
       "--to notebook --execute Parte1_EDA_KDD.ipynb "
       "--output Parte1_EDA_KDD_ejecutado.ipynb "
       "--ExecutePreprocessor.timeout=7200")
    print("\nListo:", PROY / "Parte1_EDA_KDD_ejecutado.ipynb")
else:
    print("Modo interactivo: abre http://localhost:8888 -> Parte1_EDA_KDD.ipynb -> Run All")

## 7. Monitoreo y diagnóstico

Comandos sueltos para cuando algo va lento o falla. Se pueden correr en cualquier momento, incluso mientras
el análisis está en marcha (la UI del job en vivo está en <http://localhost:4040>).

In [ ]:
_ = sh("docker compose ps", check=False)
_ = sh("docker stats --no-stream", check=False)       # CPU y RAM de cada contenedor
# sh("docker compose logs --tail=60 spark-worker", check=False)
# sh("docker compose logs --tail=60 jupyter", check=False)

### Más cores para el worker

Si la máquina tiene RAM de sobra, se pueden levantar varios workers en vez de uno (cada uno con la memoria
definida en `docker-compose.yml`):

In [ ]:
# sh("docker compose up -d --scale spark-worker=2")
# sh("docker compose restart spark-worker")   # si el worker quedó colgado

## 8. Apagar

`down` detiene y borra los contenedores. La data, el Parquet y las figuras siguen en tu disco porque viven en
la carpeta del proyecto, no dentro del contenedor.

In [ ]:
# sh("docker compose down")

---
### Alternativa sin cluster: un solo `docker run`

Para una prueba rápida, un único contenedor con Spark en modo `local[*]` (sin master ni worker). El notebook
lo detecta solo y funciona igual, solo que sin cluster:

```bash
docker run --rm -p 8888:8888 -v "$PWD":/home/jovyan/work -w /home/jovyan/work \
  quay.io/jupyter/pyspark-notebook:spark-3.5.0 \
  start-notebook.sh --IdentityProvider.token=
```

**Nota:** los comandos de Docker no funcionan *dentro* de `Parte1_EDA_KDD.ipynb`, porque ese notebook corre
adentro del contenedor y no ve el Docker de tu máquina. Por eso el manejo del entorno vive en este notebook.